# Workshop - Redis LangCache: Cache Semântico para Respostas de LLM

![Redis](https://cdn.sanity.io/images/sy1jschh/production/b4cdf336c42284d92034fffb2b14d8a79518ae82-621x456.svg)

Bem-vind[ao]s! Chamar um LLM tem custo e latência. O **LangCache** é um serviço gerenciado do Redis Cloud que guarda pares de prompt/resposta como embeddings: quando chega um prompt *semanticamente parecido* com algo já respondido, ele devolve a resposta cacheada, sem chamar o modelo de novo.

Para uma experiência completa, recomendamos abrir o **Redis Insight** (App ou Web) em paralelo, para visualizar os dados sendo criados e consultados em tempo real.

https://redis.io/insight/

## Objetivos deste notebook

- Entender o padrão *cache-aside* aplicado a um cache semântico
- Inserir respostas no cache (`set`)
- Buscar por prompts parecidos, mas não idênticos (`search`)
- Entender o efeito do `similarity_threshold` e dos atributos customizados

## Pré-requisitos

Este notebook assume que você **já tem uma cache do LangCache provisionada** no Redis Cloud. Cadastre os secrets abaixo no Colab (ícone de 🔑 **Secrets** no menu lateral esquerdo):

- `LANGCACHE_HOST`
- `LANGCACHE_CACHE_ID`
- `LANGCACHE_API_KEY`

## Setup do Ambiente

In [ ]:
!pip install -q langcache

## Setup da Conexão

In [ ]:
from google.colab import userdata

LANGCACHE_HOST = userdata.get("LANGCACHE_HOST")
LANGCACHE_CACHE_ID = userdata.get("LANGCACHE_CACHE_ID")
LANGCACHE_API_KEY = userdata.get("LANGCACHE_API_KEY")

## Teste de Conexão

In [ ]:
from langcache import LangCache

lang_cache = LangCache(
    server_url=f"https://{LANGCACHE_HOST}",
    cache_id=LANGCACHE_CACHE_ID,
    api_key=LANGCACHE_API_KEY,
)

# O LangCache não tem um "PING" dedicado - testamos credenciais e conectividade
# com uma busca simples, que deve retornar vazia (cache miss) sem lançar erro.
teste = lang_cache.search(prompt="teste de conexao", similarity_threshold=0.99)
print("Conexão ok. Resultado do teste (esperado vazio):", teste)

# Parte Um - Inserindo Respostas no Cache

Vamos simular respostas de um assistente de FAQ sobre o mesmo catálogo de produtos do notebook `search.ipynb`. Cada entrada guarda o prompt, a resposta e alguns atributos customizados (úteis para filtrar depois).

In [ ]:
entradas = [
    ("Qual o prazo de entrega para o Aspirador Robô CleanBot?",
     "O prazo de entrega padrão para o Aspirador Robô CleanBot é de 3 a 5 dias úteis.",
     {"categoria": "logistica"}),
    ("O Fone de Ouvido Bluetooth XPTO tem cancelamento de ruído?",
     "Sim, o Fone de Ouvido Bluetooth XPTO possui cancelamento de ruído ativo.",
     {"categoria": "produto"}),
    ("Posso pagar a Bicicleta Urbana CityBike em quantas vezes?",
     "A Bicicleta Urbana CityBike pode ser parcelada em até 10x sem juros.",
     {"categoria": "pagamento"}),
]

for prompt, resposta, atributos in entradas:
    lang_cache.set(prompt=prompt, response=resposta, attributes=atributos)

print(f"{len(entradas)} respostas armazenadas no cache semântico")

# Parte Dois - Buscando no Cache

## Cache HIT — mesma intenção, palavras diferentes

O prompt abaixo não é idêntico a nenhum dos que inserimos, mas tem a mesma intenção do primeiro (prazo de entrega do CleanBot).

In [ ]:
pergunta_parecida = "Em quanto tempo chega o robô de limpeza CleanBot?"

resultado = lang_cache.search(prompt=pergunta_parecida, similarity_threshold=0.9)

if resultado:
    print("Cache HIT!")
    print(resultado[0]["response"])
else:
    print("Cache MISS - precisaria chamar o LLM aqui")

## Cache MISS — pergunta sem relação com o que está cacheado

In [ ]:
pergunta_nao_relacionada = "Qual a capital da França?"

resultado = lang_cache.search(prompt=pergunta_nao_relacionada, similarity_threshold=0.9)

if resultado:
    print("Cache HIT (inesperado!)")
    print(resultado[0]["response"])
else:
    print("Cache MISS - como esperado, a pergunta não tem relação com o que está em cache")

## Filtrando por atributos customizados

Dá pra restringir a busca a uma categoria específica, mesmo dentro da mesma cache.

In [ ]:
resultado = lang_cache.search(
    prompt="Esse fone cancela ruído?",
    attributes={"categoria": "produto"},
    similarity_threshold=0.85,
)

if resultado:
    print("Cache HIT (filtrado por categoria='produto'):")
    print(resultado[0]["response"])
else:
    print("Cache MISS")

# Recapitulando

- **set:** grava um par prompt/resposta no cache semântico, com atributos opcionais
- **search:** busca por prompts semanticamente parecidos, controlando precisão com `similarity_threshold`
- **attributes:** permite restringir a busca a um subconjunto lógico da mesma cache

Este notebook faz parte de uma série de treinamento em Redis. Os outros módulos são:

- `search.ipynb` — Cache, JSON, índices e busca (textual e vetorial) com Redis Search
- `streams.ipynb` — Filas, Pub/Sub e Streams para mensageria
- `agent_memory.ipynb` — Memória de curto e longo prazo para agentes de IA
- `langcache.ipynb` — Cache semântico de respostas de LLM com LangCache